<a href="https://colab.research.google.com/github/Gusky76/Proiect-TIA/blob/work-in-progress/Project_TIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

>**Descărcarea setului de date**

Acest bloc descarcă setul de date „Plant Seedling” de pe platforma Kaggle, folosind biblioteca `kagglehub`. Setul de date conține imagini cu plante care vor fi utilizate pentru antrenarea și testarea rețelei neuronale. La final, este afișată locația în care au fost descărcate fișierele.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("siddhantsinghu/plant-seedling")

print("Path to dataset files:", path)

>**Mutarea setului de date**

Acest bloc utilizează biblioteca `shutil` pentru a muta setul de date descărcat în directorul `/content` din Google Colab. Astfel, imaginile pot fi accesate mai ușor în etapele următoare ale proiectului.

In [ ]:
import shutil
shutil.move(path, "/content" )

>**Încărcarea imaginilor**

Acest bloc încarcă imaginile pentru antrenarea și testarea rețelei neuronale, folosind biblioteca Keras. Toate imaginile sunt redimensionate la 400 × 400 pixeli pentru a avea dimensiuni uniforme. Datele sunt organizate automat în loturi (*batches*), iar clasele sunt identificate pe baza folderelor.

In [ ]:
import keras

IMG_SIZE = 400

train_data = keras.utils.image_dataset_from_directory('/content/1/train',image_size=(IMG_SIZE, IMG_SIZE))

test_data = keras.utils.image_dataset_from_directory('/content/1/test',image_size=(IMG_SIZE, IMG_SIZE))

>**Verificarea setului de date**

Acest bloc afișează informații despre seturile de date încărcate. Sunt prezentate obiectele utilizate pentru antrenare și testare, denumirile claselor de plante și numărul total al acestora. Aceste informații permit verificarea organizării setului de date.

In [ ]:
print(train_data)
print(test_data)
print(train_data.class_names)
print(len(train_data.class_names))

>**Construirea rețelei neuronale**

Acest bloc definește arhitectura unei rețele neuronale convoluționale (CNN) folosind Keras. Rețeaua conține trei straturi convoluționale pentru extragerea caracteristicilor imaginilor, urmate de straturi de pooling pentru reducerea dimensiunilor. Stratul `Flatten` transformă datele într-un vector, iar straturile `Dense` realizează clasificarea plantelor. Ultimul strat utilizează funcția `softmax` pentru a calcula probabilitățile asociate fiecărei clase.

In [ ]:
import keras
import tensorflow as tf

model = keras.Sequential([
    keras.Input(shape = (IMG_SIZE,IMG_SIZE, 3)),
    keras.layers.Conv2D(32, (3,3), activation = "relu"),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(64, (3,3), activation = "relu"),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(128, (3,3), activation = "relu"),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation = "relu"),
    keras.layers.Dense(64, activation = "relu"),
    keras.layers.Dense(32, activation = "relu"),
    keras.layers.Dense(len(train_data.class_names), activation = "softmax")
])

>**Compilarea modelului**

Acest bloc configurează modelul pentru antrenare. Optimizatorul `Adam` este utilizat pentru ajustarea parametrilor rețelei, iar funcția de pierdere `SparseCategoricalCrossentropy` măsoară erorile de clasificare. Metrica `accuracy` permite monitorizarea procentului de predicții corecte.

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

>**Antrenarea rețelei neuronale**

Acest bloc realizează antrenarea rețelei neuronale folosind imaginile din `train_data`. Procesul se desfășoară pe parcursul a 10 epoci, timp în care modelul își ajustează parametrii pentru a îmbunătăți clasificarea. Setul `test_data` este utilizat pentru validarea performanței după fiecare epocă.

In [ ]:
model.fit(
    train_data,
    validation_data=test_data,
    epochs=10
)

In [ ]:
#test